In [1]:
import sys
import os
import numpy as np
import time
import random
import pathlib

import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torchvision import transforms

from metrics import iou_score, multi_acc

from segmentation_image import SegmentationImageDataset
from segmentation_models_pytorch import Unet
import cv2

c:\Users\ADMIN\miniconda3\envs\card-detection\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
NO_OF_EPOCHS = 200
BATCH_SIZE = 16
IMAGE_SIZE = (256, 256)
COLOR_MAP = 'gray'

SEED = 50

CHECKPOINT_PATH = pathlib.Path("./model/model_checkpoint.pt")
FINAL_PATH = pathlib.Path("./model/model_final.pt")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
GEOMETRY_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomAffine(degrees = 45, translate=(0.2, 0.3), scale=(0.7, 1.1)),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
])
COLOR_TRANSFORM = None

if COLOR_MAP == 'gray':
    COLOR_TRANSFORM = transforms.Compose([
        transforms.ToTensor(),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToPILImage(),
    ])
else:
    COLOR_TRANSFORM = transforms.Compose([
        transforms.ToTensor(),
        transforms.ColorJitter(brightness=0.5, saturation=0.2, contrast=0.2, hue=0.2),
        transforms.RandomInvert(p=0.3),
        transforms.ToPILImage(),
    ])

TRANSFORM = (
    COLOR_TRANSFORM, 
    GEOMETRY_TRANSFORM,
)

NORMAL_TRANSFORM = (
    transforms.Compose([
        transforms.ToTensor(),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToPILImage(),
    ]),
    transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize(IMAGE_SIZE),
    ]),
)

In [4]:
def seed_torch(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # torch.backends.cudnn.deterministic = True

def saveCheckpoint(filename, epoch, model, optimizer, batchsize):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        "batch_size": batchsize,
    }

    # save all important stuff
    torch.save(checkpoint, filename)


In [5]:
def train(model, data_loader, criterion, optimizer, scheduler, num_epochs=5, epochs_earlystopping=10):
    logdir = './logs/' + time.strftime("%Y%m%d_%H%M%S")
    logdir = os.path.join(logdir)
    pathlib.Path(logdir).mkdir(parents=True, exist_ok=True)
    tb_writer = SummaryWriter(log_dir=logdir)

    best_F1 = 0.0
    best_loss = sys.float_info.max
    best_iou = 0.0

    early_stopping = 0

    for epoch in range(num_epochs):
        result = [f"Epoch {epoch}"]
        early_stopping += 1

        for phase in ['train', 'val']:
            if phase == 'train':  # put the model in training mode
                model.train()
            else:
                # put the model in validation mode
                model.eval()

            # keep track of training and validation loss
            batch_nums = 0
            running_loss = 0.0
            running_iou = 0.0
            running_acc = 0.0

            for (data, labels) in data_loader[phase]:
                # load the data and target to respective device
                (data, labels) = (data.to(device), labels.to(device))
                with torch.set_grad_enabled(phase == 'train'):
                    # feed the input
                    output = model(data)

                    # calculate the loss
                    loss = criterion(output, labels)

                    if phase == 'train':
                        # backward pass: compute gradient of the loss with respect to model parameters
                        print("Loss:", loss.item())
                        loss.backward()

                        optimizer.step()

                        # zero the grad to stop it from accumulating
                        optimizer.zero_grad()

                # statistics
                batch_nums += 1
                running_loss += loss.item()
                running_iou += iou_score(output, labels)
                running_acc += multi_acc(output, labels)

            if phase == 'train':
                scheduler.step(running_iou)

            # epoch statistics
            epoch_loss = running_loss / batch_nums
            epoch_iou = running_iou / batch_nums
            epoch_acc = running_acc / batch_nums

            result.append('{} Loss: {:.4f} F1 score: {:.4f} IoU: {:.4f}'.format(phase, epoch_loss, epoch_acc, epoch_iou))

            tb_writer.add_scalar('Loss/' + phase, epoch_loss, epoch)
            tb_writer.add_scalar('IoU/' + phase, epoch_iou, epoch)
            tb_writer.add_scalar('F1 score/' + phase, epoch_acc, epoch)

            if phase == 'val' and epoch_iou > best_iou:
                early_stopping = 0

                best_F1 = epoch_acc
                best_loss = epoch_loss
                best_iou = epoch_iou
                saveCheckpoint(CHECKPOINT_PATH, epoch, model, optimizer, BATCH_SIZE)
                print(
                    'Checkpoint saved - Loss: {:.4f} F1: {:.4f} IoU: {:.4f}'.format(epoch_loss, epoch_acc, epoch_iou))

        print(result)

        if early_stopping == epochs_earlystopping:
            print("Early stopped!")
            break

    print('-----------------------------------------')
    print('Final Result: Loss: {:.4f} F1 score: {:.4f}'.format(best_loss, best_F1))
    print('-----------------------------------------')

In [6]:
seed_torch()

In [7]:
print('Create datasets...')

train_dataset = SegmentationImageDataset('C:/Dataset/card/datasets/train/train/images', 'C:/Dataset/card/datasets/train/train/masks', transform=TRANSFORM)
validation_dataset = SegmentationImageDataset('C:/Dataset/card/datasets/train/val/images', 'C:/Dataset/card/datasets/train/val/masks', transform=NORMAL_TRANSFORM)

Create datasets...


In [8]:
print('Create dataloader...')
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validation_dataloader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

Create dataloader...


In [12]:

dataloader = {"train": train_dataloader,
                "val": validation_dataloader}

In [4]:
model = Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    [
        {"params": model.encoder.parameters(), "lr":1e-7},
        {"params": model.decoder.parameters(), "lr":1e-4},
    ],
    lr=1e-4,
)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.2, patience=3, verbose=True)

c:\Users\ADMIN\anaconda3\envs\card_classification\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [5]:
checkpoint = torch.load(CHECKPOINT_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_15612\2143768577.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH)


Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
      

In [15]:
train(model, dataloader, criterion, optimizer, scheduler, num_epochs=NO_OF_EPOCHS, epochs_earlystopping=10)

Loss: 0.1027560830116272
Loss: 0.10053259879350662
Loss: 0.10590856522321701
Loss: 0.10467679053544998
Loss: 0.10411380231380463
Loss: 0.09752899408340454
Loss: 0.09851380437612534
Loss: 0.10097815096378326
Loss: 0.10231448709964752
Loss: 0.10786452889442444
Loss: 0.10163858532905579
Loss: 0.09559530019760132
Loss: 0.10105331242084503
Loss: 0.09994234889745712
Loss: 0.10070747137069702
Loss: 0.10254783928394318
Loss: 0.10407420247793198
Loss: 0.11077132821083069
Loss: 0.10064443200826645
Loss: 0.10197044163942337
Loss: 0.10131344199180603
Loss: 0.0989028736948967
Loss: 0.09718477725982666
Loss: 0.09665937721729279
Loss: 0.09815596789121628
Loss: 0.10089786350727081
Loss: 0.10741197317838669
Loss: 0.1042710468173027
Loss: 0.10398580133914948
Loss: 0.09851336479187012
Loss: 0.10627481341362
Loss: 0.10153849422931671
Loss: 0.10275711119174957
Loss: 0.10179828852415085
Loss: 0.09592541307210922
Loss: 0.09960664808750153
Loss: 0.10186654329299927
Loss: 0.09973811358213425
Loss: 0.0995761603

In [47]:
torch.save(model.state_dict(), FINAL_PATH)

In [17]:
scheduler.get_last_lr()

[2e-08, 2e-05]